# Section 2: Data Preparation & Feature Engineering  --> Chapter 3: Data Extraction

---
---

**Book: Applied Machine Learning for Data Science Practitioners**
<BR>

**Author:** Vidya Subramanian (https://www.linkedin.com/in/vidyas/)


**Note:** Code is only included here for the steps that require it. Please refer to the book for the complete set of steps related to the goals covered in the chapter.

---
---

---

☑ **Install libraries**

---

In [ ]:
# # In case you need to install the relevent packages, please uncomment lines below and run (once only)

# !pip install scikit-learn==1.2.2
# !pip install pandas==2.0.3
# !pip install numpy==1.25.2
!pip install requests==2.31.0

!pip install bs4
!pip install pdfplumber==0.11.0
!pip install faker==24.14.0
!pip install IPython==7.34.0
!pip install beautifulsoup4
!pip install requests==2.31.0
!pip install petl==1.7.15
!pip install teradatasql==20.0.0.12

---

☑ **Import libraries**

---

In [ ]:
# Set some common environment variables and imports
import numpy as np  # Import numpy for numerical operations
import pandas as pd  # Import pandas for data manipulation
import os  # Import os module for accessing the file system
import csv  # Import csv module for reading CSV files

# Warnings
import warnings  # Import warnings module
warnings.filterwarnings("ignore")  # Ignore warning messages

# Setting to display all columns in a single row
pd.set_option('display.max_columns', None)  # Display all columns without truncation

# Make the Jupyter chunk window wider
from IPython.core.display import display, HTML  # Import display module from IPython
display(HTML("<style>.container { width:100% !important; }</style>"))  # Set display width to 100%

from pydrive2.auth import GoogleAuth  # Import GoogleAuth class from the pydrive.auth module
from pydrive2.drive import GoogleDrive # Import GoogleDrive class from the pydrive.drive module
from google.colab import auth  # Import auth class from the google.colab module
from oauth2client.client import GoogleCredentials  # Import GoogleCredentials class from the oauth2client.client module

# import necessary libraries
# import teradatasql
import petl as etl
import requests  # Import requests library for making HTTP requests
from bs4 import BeautifulSoup  # Import BeautifulSoup for HTML parsing
import pdfplumber  # Import pdfplumber for extracting text from PDF

from faker import Faker  # Import Faker for generating fake data
from random import randrange  # Import randrange for generating random numbers
from datetime import datetime  # Import datetime for working with dates

# Initialize logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


### 1.3. Structured Data Extraction

#####**Step 2b:** Extract the structured data from the data source(s).

In [ ]:
# ***** Function to help display*****
def print_pretty_header(title, subtitle):
    # Define header formatting
    line_length = 50  # Define line length for header
    header_padding = 2  # Define padding for header

    # Calculate the width for the title text
    title_width = line_length  * header_padding  # Calculate title width based on padding

    # Print top border
    print("#" * title_width + "\n")

    # Print centered title
    print(title.center(title_width) + "\n")

    # If there's a subtitle, print it
    if subtitle:
        # Calculate the width for the subtitle text
        subtitle_width = line_length - 2 * header_padding  # Calculate subtitle width based on padding
        print(subtitle.center(title_width)+ "\n")

    # Print bottom border
    print("#" * title_width + "\n")


#### Read from Google Colab

If you use a different file, then to run the code successfully:

(1) Right click on the shared drive file and click on share --> Copy.

(2) When you paste the link, it looks like = "https://drive.google.com/file/d/1ZzpYsHLSoHgwKc9HnDIBaFuM74b2J2h0/view?usp=drive_link"

(3) Copy the id between "https://drive.google.com/file/d/" and "/view?usp=drive_link" from "https://drive.google.com/file/d/1ZzpYsHLSoHgwKc9HnDIBaFuM74b2J2h0/view?usp=drive_link". Here our shared ID is "1ZzpYsHLSoHgwKc9HnDIBaFuM74b2J2h0".

(4) Replace the id after the = in the following statement with the shared link code
dataLink = 'https://drive.google.com/open?id=1ZzpYsHLSoHgwKc9HnDIBaFuM74b2J2h0'

In [ ]:
def get_data(colabFileName, fileLink):
    try:
        # Authenticate and create the PyDrive client.
        auth.authenticate_user()  # Authenticate the user
        gauth = GoogleAuth()  # Create a GoogleAuth instance
        gauth.credentials = GoogleCredentials.get_application_default()  # Use the default application credentials
        drive = GoogleDrive(gauth)  # Create a GoogleDrive instance using the authenticated GoogleAuth instance

        # Read File
        ignorestr, id = fileLink.split('=')  # Split the link by '=' and get the id of the file
        downloaded = drive.CreateFile({'id':id})  # Create a PyDrive File instance with the specified file id
        downloaded.GetContentFile(colabFileName)  # Download the file and save it to the local file system
        df = pd.read_csv(colabFileName)  # Read the CSV file into a Pandas dataframe
        return df  # Return the Pandas dataframe
    except Exception as e:
        logging.error(f"Failed to get data: {e}")
        raise
        return None

# Read the file and display contents.
print_pretty_header("Example of Data Extraction", "All Data")
colabFileName = 'S2_Ch3_Data_Extraction_data.csv'
dataLink = 'https://drive.google.com/open?id=1ZzpYsHLSoHgwKc9HnDIBaFuM74b2J2h0'  # The shareable link to the file
df_rawdata = get_data(colabFileName, dataLink)  # Call the get_data() function
df_rawdata_org = df_rawdata.copy()  # Make a copy of the Pandas dataframe for ease of dropping columns
df_rawdata  # Display the Pandas dataframe containing the training data.

####################################################################################################

                                     Example of Data Extraction                                     

                                              All Data                                              

####################################################################################################



,VacationHomeID,VacHomeClass,VacHomeZone,VacHomeLotFrontage,VacHomeLotSqFt,VacHomeStreet,VacHomeRoad,VacHomeLotShape,VacHomeLandContour,VacHomeUtilities,VacHomeLotConfig,VacHomeLandSlope,VacHomeNeighborhood,VacHomeCondition,VacHomeBuilding,VacHomeHouseStyle,VacHomeRating,VacHomeQuality,VacHomeConsYear,VacHomeSince,VacHomeRoofStyle,VacHomeRoofMat,VacHomeExterior,VacHomeMasonryVeneer,VacHomeMasonryArea,VacHomeExterQual,VacHomeQual,VacHomeFoundation,VacHomeBsmtQuality,VacHomeBsmtLight,VacHomeBsmtFinish,VacHomeBsmtSqFt,VacHomeHeating,VacHomeHeatingQuality,VacHomeAC,VacHomeElectricalWiring,VacHomeFloor1SqFt,VacHomeFloor2SqFt,VacHomeSqFt,VacHomeNumFullBath,VacHomeNumHalfBath,VacHomeBedroomWithCloset,VacHomeKitcheninHome,VacHomeKitchenQuality,VacHomeRooms,VacHomeFireplaces,VacHomeFireplaceQuality,VacHomeGarageType,VacHomeGarageFinish,VacHomeCarsInGarage,VacHomeGarageArea,VacHomeGarageQuality,VacHomeDriveway,VacHomeWoodDeckSqFt,VacHomePorchSqFt,VacHomePoolSqFt,VacHomePoolQuality,VacHomeFence,VacHomeStartMonth,VacHomeStartYear,VacHomeSaleType,VacHomeSaleCondition,VacHomeSalePrice,VacHomeInterestInHome,VacHomeGoodSchools,VacHomeAvailableDate,VacHomeBarCode,VacHomeOwnerAddress,VacHomeOwnerCity,VacHomeOwnerCountry,VacHomeOwnerEmail,VacHomeOwnerGender,VacHomeOwnerState,VacHomeOwnerZipcode,VacHomeRenovationAmount,VacHomeSurveyDate,VacHomeSurveyRating
0,1,VacHomeClass-6,VacHomeZone-1,65.0,8450,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 6,Proximity to School,Apartment,Bungalow,7,5,2008,2009,Gable,Rolled Roofing,Board Batten,Natural Stone,196.0,4,3,Individual footing,4.0,No Sunlight,ZenWall Panels,856,Furnace,5,Yes,NM Cable,856,854,1710,2,1,3,1,4,8,0,NaN,Attached,Wood Sheathing,2,548,3,Paved,0,0,0,0,No Fence,2,2008,Down payment assistance,Regular,173061.00,Y,Y,2021-11-24,78408012,Jochen-Peukert-Straße 4/7\n93824 Wolgast,Schwandorf,Venezuela,sylvester59@haase.de,M,Niedersachsen,41157,93.0,2021-11-24,4.6
1,2,VacHomeClass-1,VacHomeZone-1,80.0,9600,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,T-intersection,No Slope,RNH 25,Proximity to Hospital,Apartment,Contemporary,6,8,1977,1979,Gable,Rolled Roofing,Log Wood,Manmade Stone,0.0,3,3,Combined footing,4.0,French Doors,BrightWall Paneling,1262,Furnace,5,Yes,NM Cable,1262,0,1262,2,0,3,1,3,6,1,TA,Attached,Wood Sheathing,2,460,3,Paved,298,0,0,0,No Fence,5,2007,Down payment assistance,Regular,150651.00,N,Y,2021-10-03,23209473,Henkweg 1\n93328 Bremen,Hersbruck,Bolivien,tilman19@junck.org,F,Thüringen,84185,9.0,2021-01-16,3.8
2,3,VacHomeClass-6,VacHomeZone-1,68.0,11250,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 6,Proximity to School,Apartment,Bungalow,7,5,2006,2008,Gable,Rolled Roofing,Board Batten,Natural Stone,162.0,4,3,Individual footing,4.0,Small Windows,ZenWall Panels,920,Furnace,5,Yes,NM Cable,920,866,1786,2,1,3,1,4,6,1,TA,Attached,Wood Sheathing,2,608,3,Paved,0,0,0,0,No Fence,9,2008,Down payment assistance,Regular,185511.00,Y,N,2021-09-21,12309894,Mercedes-Butte-Allee 8/4\n39694 Neustrelitz,Borken,Pakistan,erich35@kade.com,M,Rheinland-Pfalz,79533,88.0,2021-07-08,4.4
3,4,VacHomeClass-7,VacHomeZone-1,60.0,9550,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,Corner lot,No Slope,RNH 7,Proximity to School,Apartment,Bungalow,7,5,1916,1975,Gable,Rolled Roofing,Wood Shingle,Manmade Stone,0.0,3,3,Strip foundation,3.0,No Sunlight,BrightWall Paneling,756,Furnace,4,Yes,NM Cable,961,756,1717,1,0,3,1,4,7,1,Gd,Detached,Metal Panels,3,642,3,Paved,0,0,0,0,No Fence,2,2006,Down payment assistance,Shortsale,116206.00,Y,N,2021-01-11,9163454,Remo-Conradi-Ring 090\n21730 Potsdam,Schwarzenberg,Thailand,burkardpaertzelt@googlemail.com,F,Brandenburg,13145,94.0,2021-07-05,3.9
4,5,VacHomeClass-6,VacHomeZone-1,84.0,14260,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,T-intersection,No Slope,RNH 14,Proximity to School,Apartment,Bungalow,8,5,2001,2006,Gable,Rolled Roofing,Board Batten,Natural Stone,350.0

In [ ]:
# Get the number of rows and columns in the DataFrame
rows = len(df_rawdata.axes[0])  # Get the number of rows by accessing the length of the first axis
cols = len(df_rawdata.axes[1])  # Get the number of columns by accessing the length of the second axis

# Print the number of rows and columns
print_pretty_header("Number of Rows and Columns in Training Data", "")  # Print header for better visualization
print("Number of Rows: " + str(rows))  # Print the number of rows
print("Number of Columns: " + str(cols))  # Print the number of columns

####################################################################################################

                            Number of Rows and Columns in Training Data                             

####################################################################################################

Number of Rows: 1460
Number of Columns: 77


#### Read a CSV File from your local machine.
Including this code for completeness. Its useful to have code to read a CSV file from the hard drive if you are using a tool other than Colab. Don't forget to uncomment the code and add your filename.

Colab requires files to be uploaded before they can be read. So, if you are using Colab, skip to (d) below.

In [ ]:
# def readFromCSV(csv_filename):
#     try:
#         # Print the current working directory
#         print("Current working directory: {0}".format(os.getcwd()))

#         # Open the CSV file in read mode
#         with open(csv_filename, mode='r') as file:
#             # Create a CSV reader object
#             csv_reader = csv.reader(file)

#             # Iterate over each row in the CSV file
#             for row in csv_reader:
#                 print(row)  # Print the row

#     except Exception as e:
#         logging.error(f"Error reading CSV File: {e}")
#         return ""


# # Specify the name of the CSV file to read
# csv_filename = 'Please Put Your CSV Filename here'

# # Call the readFromCSV function with the specified filename
# readFromCSV(csv_filename)

#### Read an Excel File
Including this code for completeness. Its useful to have code to read an Excel file from the hard drive if you are using a tool other than Colab. Don't forget to uncomment the code and add your filename.

Colab requires files to be uploaded before they can be read. So, if you are using Colab, skip to (d) below.

In [ ]:
def readFromExcel(excel_filename):
    try:
        # Print the current working directory
        print("Current working directory: {0}".format(os.getcwd()))

        # Read by default the 1st sheet of an excel file
        df_excel = pd.read_excel(excel_filename)  # Read Excel file into a Pandas DataFrame
        return df_excel  # Return the DataFrame
    except Exception as e:
        logging.error(f"Error reading excel file: {e}")
        return ""

# Make sure your file is in the current directory or specify the path in the filename below
excel_filename = 'Put Your XLS Filename here'

# Call the readFromExcel function to read the Excel file
df_excel = readFromExcel(excel_filename)

# Print the DataFrame
print(df_excel)

ERROR:root:Error reading excel file: [Errno 2] No such file or directory: 'Put Your XLS Filename here'


Current working directory: /content



#### Read from Teradata
Including this code for completeness. Its useful to have code to read from Teradata or any RDBMS. Don't forget to uncomment the code before you run it.

Colab requires files to be uploaded before they can be read. So, if you are using Colab, skip to (d) below.

In [ ]:
# def readFromTeradata():
#     # establish a connection to a Teradata database using teradatasql
#     conn = teradatasql.connect(host='localhost', user='myuser', password='mypassword', database='mydatabase')

#     # define an SQL query
#     sql = "SELECT ColName_1, ColName_2 FROM TableName WHERE Filter = 'FilterValue'"

#     # extract data from the database using petl.fromdb
#     with conn.cursor() as cur:
#         cur.execute(sql)
#         table1 = etl.fromdicts(cur.fetchall())

#     # sort the data by ColName_2 using petl.sort
#     table2 = etl.sort(table1, 'ColName_2')

#     # save the sorted data to a CSV file using petl.tocsv
#     etl.tocsv(table2, 'abcd_data.csv')

#     # define a new table with columns ColName_1 and ColName_2 and some sample data
#     table3 = [['ColName_1', 'ColName_2'],
#               ['NewProductId1', 'NewProductName1'],
#               ['NewProductId2', 'NewProductName2']]

#     # append the new data to the database using petl.appenddb
#     with conn.cursor() as cur:
#         for row in table3[1:]:
#             cur.execute("INSERT INTO TableName (ColName_1, ColName_2) VALUES (?, ?)", row)

#     conn.commit()
#     conn.close()

# readFromTeradata()

### 1.4. Semi-structured Data Extraction

#####**Step 3a:** Extract semi-structured using web crawling
Including this code for completeness. We do not use this data in our example. Its only for academic learning.

In [ ]:
def testwebcrawler():
    try:

        # Define the URL to scrape
        url = "https://en.wikipedia.org/wiki/Main_Page"

        # Send a request to the website and get the HTML content
        response = requests.get(url)  # Send GET request to the URL
        html_content = response.content  # Get the HTML content of the response

        # Parse the HTML content with BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')  # Parse HTML content

        # Find all the links in the HTML content
        links = soup.find_all('a')  # Find all <a> tags (links) in the HTML content

        # Print the first 5 links found
        # Print the number of rows and columns
        print_pretty_header("List of Pages from Web Crawling", "")  # Print header
        for link in links[:6]:  # Iterate over the first 6 links
            # Check if the link points to a specific location on the same page
            if link.get('href') != "#bodyContent":
                # Print the link's URL
                print(link.get('href'))
    except Exception as e:
        logging.error(f"Error with web crawler: {e}")
        return ""

# web crawling
testwebcrawler()  # Call the function to execute the web crawling

####################################################################################################

                                  List of Pages from Web Crawling                                   

####################################################################################################

/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About


#####**Step 3b:** Extract semi-structured using web scraping
Including this code for completeness. We do not use this data in our example. Its only for academic learning.

In [ ]:
def webscrapetest():
    try:
        # Specify the URL of the search results page
        url = "https://en.wikipedia.org/w/index.php?search=machine+learning+books&title=Special:Search&ns0=1&searchToken=54wp911b1al8hx1p1ahsoqfax"

        # Send a GET request to the URL and get the HTML content
        response = requests.get(url)  # Send GET request to the URL
        html_content = response.content  # Get the HTML content of the response

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')  # Parse HTML content

        # Find the div containing the search results
        search_results_div = soup.find('div', {'class': 'searchresults'})  # Find div with class 'searchresults'

        # Find all the links to the book pages in the search results
        book_links = search_results_div.find_all('td', {'class': 'searchResultImage-thumbnail'})  # Find all 'td' elements with class 'searchResultImage-thumbnail'
        counter = 0  # Initialize counter for book number

        # Print the title and URL of each book
        print_pretty_header("List of Books from Web Scraping", "")  # Print header
        for td in book_links:  # Iterate over each 'td' element in book_links
            for a in td:  # Iterate over each 'a' element in 'td'
                counter += 1  # Increment counter for each book
                # Check if the link has a 'title' attribute, and if so, print it as the book title
                if a.get('title'):
                    book_title = a.get('title')  # Get book title from 'title' attribute
                    print(book_title)  # Print book title
    except Exception as e:
        logging.error(f"Error web scraping: {e}")
        return ""

# Webscrape
webscrapetest()  # Call the function to execute the web scraping


####################################################################################################

                                  List of Books from Web Scraping                                   

####################################################################################################



### 1.5. Unstructured Data Extraction

#####**Step 4a:** Read contents from a PDF File Example
Including this code for completeness. We do not use this data in our example. Its only for academic learning.

In [ ]:
def extract_text_from_pdf(pdf_file_path):
    try:
        # Open the PDF file
        with pdfplumber.open(pdf_file_path) as pdf:
            text = ""  # Initialize variable to store extracted text
            # Iterate through each page in the PDF
            for page in pdf.pages:
                # Extract text from the current page
                page_text = page.extract_text()
                # Append the text from the current page to the result
                text += page_text

        return text  # Return the extracted text
    except Exception as e:
        logging.error(f"Failed to get data: {e}")
        raise

# Authenticate and create the PyDrive client.
auth.authenticate_user()  # Authenticate the user
gauth = GoogleAuth()  # Create a GoogleAuth instance
gauth.credentials = GoogleCredentials.get_application_default()  # Use the default application credentials
drive = GoogleDrive(gauth)  # Create a GoogleDrive instance using the authenticated GoogleAuth instance

# Specify the path to your PDF file
pdf_file_path = "S2_Ch3_Data_Extraction_unstructured_PDF.pdf"

pdf_link = 'https://drive.google.com/open?id=1a9_SBOfirgFowJm7Av37S1qFJa0dXBPe'  # The shareable link to the file
ignorestr, id = pdf_link.split('=')  # Split the link by '=' and get the id of the file
downloaded = drive.CreateFile({'id':id})  # Create a PyDrive File instance with the specified file id
downloaded.GetContentFile(pdf_file_path)  # Download the file and save it to the local file system

# Extract text from the PDF
extracted_text = extract_text_from_pdf(pdf_file_path)

# Print the extracted text
print_pretty_header("Printing Contents of an unstructured PDF File", "")  # Print header
print(extracted_text)  # Print extracted text


####################################################################################################

                           Printing Contents of an unstructured PDF File                            

####################################################################################################

Dummy PDF file


### 1.6. Synthetic Data Creation

#####**Step 5a:** Create synthetic data to train the model

In [ ]:
def createSampleReviewData():
    try:
        # Create a copy of a DataFrame ID column
        df_VacHomeOwnerReviewData = df_rawdata.VacationHomeID.copy()

        # Get the number of rows in the DataFrame
        df_VacHomeOwnerReviewData_idx = len(df_VacHomeOwnerReviewData.index)

        # Create a Faker object for generating fake data
        fake = Faker('de_DE')  # Initialize Faker with German locale

        # Set the seed for Faker to get consistent results
        Faker.seed(0)  # Set seed to 0 for reproducibility

        # Generate fake review data for each vacation home ID
        VacHome_home = []
        for VacHome_home_id in range(df_VacHomeOwnerReviewData_idx):

            # Create a random review date between Jan 1, 2021 and Dec 31, 2021
            d1 = datetime.strptime(f'1/1/2021', '%m/%d/%Y')  # Start date
            d2 = datetime.strptime(f'12/31/2021', '%m/%d/%Y')  # End date
            VacHomeReviewDate = fake.date_between(d1, d2)  # Generate random date between d1 and d2

            # Create a random review rating between 1 and 5 with one decimal place
            VacHomeReviewRating = fake.pyfloat(right_digits=1, positive=True, min_value=1, max_value=5)  # Generate random float between 1 and 5

            # Add the review data to the list for this vacation home
            VacHome_home.append([VacHome_home_id , VacHomeReviewDate, VacHomeReviewRating])

        # Create a new DataFrame with the review data
        df_VacHomeOwnerReviewData_updated = pd.DataFrame(VacHome_home, columns=['VacationHomeID', 'VacHomeReviewDate', 'VacHomeReviewRating'])

        # Set the Pandas option to display all columns
        pd.pandas.set_option('display.max_columns', None)  # Set display option to show all columns

        return df_VacHomeOwnerReviewData_updated  # Return the new DataFrame with synthetic review data
    except Exception as e:
        logging.error(f"Failed to get data: {e}")
        raise

# Call the createSampleReviewData function
df_VacHomeOwnerReviewData_updated = createSampleReviewData()  # Generate synthetic review data

print_pretty_header("Synthetic Data Created", "")  # Print header


####################################################################################################

                                       Synthetic Data Created                                       

####################################################################################################



### 1.7. Data Integration (Optional)

#####**Step 6a:**  Learn how to integrate disjoint sets of data.


In [ ]:
def integratedataframes():
    try:
        # Select columns in the updated DataFrame that are not in the raw DataFrame
        cols_to_use = df_VacHomeOwnerReviewData_updated.columns.difference(df_rawdata.columns)

        # Merge the two DataFrames based on their indices, using the selected columns
        df_final_result = pd.merge(df_rawdata, df_VacHomeOwnerReviewData_updated[cols_to_use], left_index=True, right_index=True, how='outer')

        # Set the file name
        file_name = "S2_Ch3_Data_Extraction_Integrated_data.csv"

        # Write the DataFrame to a CSV file
        df_final_result.to_csv(file_name, sep=',', encoding='utf-8', index=False)

        # Download the file from the Colab VM to the local PC
        from google.colab import files
        files.download(file_name)

        # Return the merged DataFrame
        return df_final_result

    except Exception as e:
        logging.error(f"Failed to get data: {e}")
        raise
        return None

# Call the function to merge the DataFrames
df_final_result = integratedataframes()

# Show the first 5 rows of the merged DataFrame
print_pretty_header("Data Integration Example", "")
print("Check your Downloads Folder for file - S2_Ch3_Data_Extraction_Integrated_data.csv.")
df_final_result.tail(5)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

####################################################################################################

                                      Data Integration Example                                      

####################################################################################################

Check your Downloads Folder for file - S2_Ch3_Data_Extraction_Integrated_data.csv.


,VacationHomeID,VacHomeClass,VacHomeZone,VacHomeLotFrontage,VacHomeLotSqFt,VacHomeStreet,VacHomeRoad,VacHomeLotShape,VacHomeLandContour,VacHomeUtilities,VacHomeLotConfig,VacHomeLandSlope,VacHomeNeighborhood,VacHomeCondition,VacHomeBuilding,VacHomeHouseStyle,VacHomeRating,VacHomeQuality,VacHomeConsYear,VacHomeSince,VacHomeRoofStyle,VacHomeRoofMat,VacHomeExterior,VacHomeMasonryVeneer,VacHomeMasonryArea,VacHomeExterQual,VacHomeQual,VacHomeFoundation,VacHomeBsmtQuality,VacHomeBsmtLight,VacHomeBsmtFinish,VacHomeBsmtSqFt,VacHomeHeating,VacHomeHeatingQuality,VacHomeAC,VacHomeElectricalWiring,VacHomeFloor1SqFt,VacHomeFloor2SqFt,VacHomeSqFt,VacHomeNumFullBath,VacHomeNumHalfBath,VacHomeBedroomWithCloset,VacHomeKitcheninHome,VacHomeKitchenQuality,VacHomeRooms,VacHomeFireplaces,VacHomeFireplaceQuality,VacHomeGarageType,VacHomeGarageFinish,VacHomeCarsInGarage,VacHomeGarageArea,VacHomeGarageQuality,VacHomeDriveway,VacHomeWoodDeckSqFt,VacHomePorchSqFt,VacHomePoolSqFt,VacHomePoolQuality,VacHomeFence,VacHomeStartMonth,VacHomeStartYear,VacHomeSaleType,VacHomeSaleCondition,VacHomeSalePrice,VacHomeInterestInHome,VacHomeGoodSchools,VacHomeAvailableDate,VacHomeBarCode,VacHomeOwnerAddress,VacHomeOwnerCity,VacHomeOwnerCountry,VacHomeOwnerEmail,VacHomeOwnerGender,VacHomeOwnerState,VacHomeOwnerZipcode,VacHomeRenovationAmount,VacHomeSurveyDate,VacHomeSurveyRating,VacHomeReviewDate,VacHomeReviewRating
1455,1456,VacHomeClass-6,VacHomeZone-1,62.0,7917,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 9,Proximity to School,Apartment,Bungalow,6,5,2004,2006,Gable,Rolled Roofing,Board Batten,Manmade Stone,0.0,3,3,Individual footing,4.0,No Sunlight,Dry Wall,953,Furnace,5,Yes,NM Cable,953,694,1647,2,1,3,1,3,7,1,TA,Attached,Wood Sheathing,2,460,3,Paved,0,0,0,0,No Fence,8,2007,Down payment assistance,Regular,145256.00,N,Y,2021-09-15,21425578,Guteplatz 1\n74470 Hohenmölsen,Chemnitz,Isle of Man,erdal97@schuchhardt.de,F,Berlin,5310,61.0,2021-05-08,3.5,2021-10-22,2.2
1456,1457,VacHomeClass-1,VacHomeZone-1,85.0,13175,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 17,Proximity to School,Apartment,Contemporary,6,6,1983,1995,Gable,Rolled Roofing,Brick,Manmade Panels,119.0,3,3,Combined footing,4.0,No Sunlight,BrightWall Paneling,1542,Furnace,3,Yes,NM Cable,2073,0,2073,2,0,3,1,3,7,2,TA,Attached,Metal Panels,2,500,3,Paved,349,0,0,0,Vinyl Fences,2,2010,Down payment assistance,Regular,174306.00,N,Y,2021-04-22,69114847,Fischergasse 3/3\n37451 Hechingen,Flöha,Katar,kuehnertanna-luise@dippel.com,M,Sachsen-Anhalt,42804,31.0,2021-11-15,1.4,2021-06-30,4.0
1457,1458,VacHomeClass-7,VacHomeZone-1,66.0,9042,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 7,Proximity to School,Apartment,Bungalow,7,9,1944,2010,Gable,Rolled Roofing,Metal,Manmade Stone,0.0,5,4,Drilled Shafts,3.0,No Sunlight,ZenWall Panels,1152,Furnace,5,Yes,NM Cable,1188,1152,2340,2,0,4,1,4,9,2,Gd,Attached,Wood Sheathing,1,252,3,Paved,0,0,0,0,Composite Fences,5,2010,Down payment assistance,Regular,221201.00,Y,Y,2021-07-13,46752383,Adina-Drewes-Gasse 6\n44492 Apolda,Altötting,Svalbard und Jan Mayen,eminotto@gmail.com,Unknown,Hessen,1482,60.0,2021-08-08,2.5,2021-09-26,4.3
1458,1459,VacHomeClass-1,VacHomeZone-1,68.0,9717,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,NAmes,Proximity to School,Apartment,Contemporary,5,6,1951,1999,Sawtooth,Rolled Roofing,Log Wood,Manmade Stone,0.0,3,3,Combined footing,3.0,Small Windows,ZenWall Panels,1078,Furnace,4,Yes,Underground Feeder Cable,1078,0,1078,1,0,2,1,4,5,0,NaN,Attached,Metal Panels,1,240,3,Paved,366,0,0,0,No Fence,4,2010,Down payment assistance,Regular,117969.75,Y,Y,2021-11-13,41700105,Michaela-Bauer-Weg 0\n29985 Genthin,Neustrelitz,Kuwait,sina70@karge.net,M,Rheinland-Pfalz,67129,62.0,2021-01-15,1.9,2021-07-11,4.1
1459,1460,VacHomeClass-1,VacHomeZone-1,75.0,9937,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 8,Proximit

------------- End of **S2_Ch3_Data_Extraction Chapter Code** for **Applied Machine Learning for Data Science Practitioners** ------------ Vidya Subramanian ------------------